## Import

In [12]:
import pandas as pd
import numpy as np
import pickle
from moviepy.editor import VideoFileClip
import os
import librosa
import librosa.display
import matplotlib.pyplot as plt
import parselmouth
from IPython.display import Audio, display
from librosa import display
import datetime
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import xgboost as xgb
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

from sklearn.model_selection import GridSearchCV
import keras_tuner as kt
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense
from keras.optimizers import Adam
import joblib

# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc

## Preprocessing

In [47]:
def resampling(y: np.ndarray, sr: int, target_sr: int = 16000) -> Tuple[np.ndarray, int]:
    y_resampled = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
    return y_resampled, target_sr

In [48]:
def mmse_denoise(noisy_signal: np.ndarray) -> np.ndarray:
    stft_signal = librosa.stft(noisy_signal)
    noise_psd = np.mean(np.abs(stft_signal[:, :10]) ** 2, axis=1, keepdims=True)
    speech_psd = np.abs(stft_signal) ** 2
    gain = np.maximum(1 - noise_psd / (speech_psd + 1e-10), 0)
    enhanced_stft = gain * stft_signal
    enhanced_signal = librosa.istft(enhanced_stft)
    return enhanced_signal

In [49]:
def normalize_audio(audio: np.ndarray) -> np.ndarray:
    # Ensure the audio signal has non-zero elements to avoid division by zero
    if np.max(np.abs(audio)) == 0:
        return audio  # If the audio is silent, return it as-is
    
    # Normalize audio by dividing by the maximum absolute value
    normalized_audio = audio / np.max(np.abs(audio))
    return normalized_audio

In [50]:
# Function to get statistical summaries
def get_statistics(feature):
    if feature.size == 0 or np.all(np.isnan(feature)):
        return {"mean": 0, "min": 0, "max": 0, "std_dev": 0}
    return {
        'mean': np.nanmean(feature),
        'min': np.nanmin(feature),
        'max': np.nanmax(feature),
        'std_dev': np.nanstd(feature)
    }

In [51]:
def extract_audio_features(y: np.ndarray, sr: int) -> pd.DataFrame:
    # Load the audio into PRAAT's Sound object
    sound = parselmouth.Sound(y, sampling_frequency=sr)

     # Extract intensity and pitch objects for prosodic features
    intensity_obj = sound.to_intensity()
    pitch_obj = sound.to_pitch(pitch_floor=75, pitch_ceiling=600)
    
    # Prosodic features
    # Intensity
    intensity_values = intensity_obj.values.flatten()
    intensity_stats = get_statistics(intensity_values)

    # Pitch
    pitch_values = pitch_obj.selected_array['frequency']
    pitch_values = pitch_values[pitch_values > 0]  # Filter unvoiced frames
    pitch_stats = get_statistics(pitch_values)

    # Energy
    energy = np.sum(y ** 2)

    # Speech rate (zero crossings per second)
    zero_crossings = np.sum(np.abs(np.diff(np.sign(y))) > 0)
    duration = sound.get_total_duration()
    speech_rate = zero_crossings / duration

    # Spectral features
    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_stats = [get_statistics(mfcc[i]) for i in range(mfcc.shape[0])]

    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_stats = [get_statistics(chroma[i]) for i in range(chroma.shape[0])]

    # Spectral contrast
    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    spectral_contrast_stats = [get_statistics(spectral_contrast[i]) for i in range(spectral_contrast.shape[0])]

    # Spectral centroid
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spectral_centroid_stats = get_statistics(spectral_centroid[0])

    # Spectral bandwidth
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    spectral_bandwidth_stats = get_statistics(spectral_bandwidth[0])

    # Spectral flatness
    spectral_flatness = librosa.feature.spectral_flatness(y=y)
    spectral_flatness_stats = get_statistics(spectral_flatness[0])

    # Spectral roll-off
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)
    spectral_rolloff_stats = get_statistics(spectral_rolloff[0])

    # LPC (Linear Predictive Coding)
    lpc_coeffs = librosa.lpc(y, order=16)
    lpc_stats = get_statistics(lpc_coeffs)


     # Voice Quality features
    # Formants (F1, F2, F3)
    formant = sound.to_formant_burg()
    f1 = np.array([formant.get_value_at_time(1, t) for t in np.linspace(0, duration, 100)])
    f1_stats = get_statistics(f1)
    f2 = np.array([formant.get_value_at_time(2, t) for t in np.linspace(0, duration, 100)])
    f2_stats = get_statistics(f2)
    f3 = np.array([formant.get_value_at_time(3, t) for t in np.linspace(0, duration, 100)])
    f3_stats = get_statistics(f3)

    # Harmonic-to-Noise Ratio (HNR)
    hnr = sound.to_harmonicity()
    hnr_values = hnr.values.flatten()
    hnr_stats = get_statistics(hnr_values)

    # RMS energy
    rms = librosa.feature.rms(y=y)
    rms_stats = get_statistics(rms[0])

    # Zero Crossing Rate
    zero_crossing_rate = librosa.feature.zero_crossing_rate(y=y)
    zero_crossing_rate_stats = get_statistics(zero_crossing_rate[0])



    # Compile features into a DataFrame
    features = {
        # Prosodic features
        'intensity_mean': intensity_stats['mean'],
        'intensity_min': intensity_stats['min'],
        'intensity_max': intensity_stats['max'],
        'intensity_std_dev': intensity_stats['std_dev'],
        'pitch_mean': pitch_stats['mean'],
        'pitch_min': pitch_stats['min'],
        'pitch_max': pitch_stats['max'],
        'pitch_std_dev': pitch_stats['std_dev'],
        'energy': energy,
        'speech_rate': speech_rate,

        # Spectral features
        'spectral_centroid_mean': spectral_centroid_stats['mean'],
        'spectral_centroid_min': spectral_centroid_stats['min'],
        'spectral_centroid_max': spectral_centroid_stats['max'],
        'spectral_centroid_std_dev': spectral_centroid_stats['std_dev'],
        'spectral_bandwidth_mean': spectral_bandwidth_stats['mean'],
        'spectral_bandwidth_min': spectral_bandwidth_stats['min'],
        'spectral_bandwidth_max': spectral_bandwidth_stats['max'],
        'spectral_bandwidth_std_dev': spectral_bandwidth_stats['std_dev'],
        'spectral_flatness_mean': spectral_flatness_stats['mean'],
        'spectral_flatness_min': spectral_flatness_stats['min'],
        'spectral_flatness_max': spectral_flatness_stats['max'],
        'spectral_flatness_std_dev': spectral_flatness_stats['std_dev'],
        'spectral_rolloff_mean': spectral_rolloff_stats['mean'],
        'spectral_rolloff_min': spectral_rolloff_stats['min'],
        'spectral_rolloff_max': spectral_rolloff_stats['max'],
        'spectral_rolloff_std_dev': spectral_rolloff_stats['std_dev'],
        **{f'mfcc_{i}_mean': mfcc_stats[i]['mean'] for i in range(13)},
        **{f'mfcc_{i}_min': mfcc_stats[i]['min'] for i in range(13)},
        **{f'mfcc_{i}_max': mfcc_stats[i]['max'] for i in range(13)},
        **{f'mfcc_{i}_std_dev': mfcc_stats[i]['std_dev'] for i in range(13)},
        **{f'chroma_{i}_mean': chroma_stats[i]['mean'] for i in range(12)},
        **{f'chroma_{i}_min': chroma_stats[i]['min'] for i in range(12)},
        **{f'chroma_{i}_std_dev': chroma_stats[i]['std_dev'] for i in range(12)},
        **{f'spectral_contrast_{i}_mean': spectral_contrast_stats[i]['mean'] for i in range(7)},
        **{f'spectral_contrast_{i}_min': spectral_contrast_stats[i]['min'] for i in range(7)},
        **{f'spectral_contrast_{i}_max': spectral_contrast_stats[i]['max'] for i in range(7)},
        **{f'spectral_contrast_{i}_std_dev': spectral_contrast_stats[i]['std_dev'] for i in range(7)},
        'lpc_mean': lpc_stats['mean'],
        'lpc_min': lpc_stats['min'],
        'lpc_max': lpc_stats['max'],
        'lpc_std_dev': lpc_stats['std_dev'],

        # Voice Quality features
        'f1_mean': f1_stats['mean'],
        'f1_min': f1_stats['min'],
        'f1_max': f1_stats['max'],
        'f1_std_dev': f1_stats['std_dev'],
        'f2_mean': f2_stats['mean'],
        'f2_min': f2_stats['min'],
        'f2_max': f2_stats['max'],
        'f2_std_dev': f2_stats['std_dev'],
        'f3_mean': f3_stats['mean'],
        'f3_min': f3_stats['min'],
        'f3_max': f3_stats['max'],
        'f3_std_dev': f3_stats['std_dev'],
        'hnr_mean': hnr_stats['mean'],
        'hnr_min': hnr_stats['min'],
        'hnr_max': hnr_stats['max'],
        'hnr_std_dev': hnr_stats['std_dev'],
        'rms_mean': rms_stats['mean'],
        'rms_min': rms_stats['min'],
        'rms_max': rms_stats['max'],
        'rms_std_dev': rms_stats['std_dev'],
        'zero_crossing_rate_mean': zero_crossing_rate_stats['mean'],
        'zero_crossing_rate_min': zero_crossing_rate_stats['min'],
        'zero_crossing_rate_max': zero_crossing_rate_stats['max'],
        'zero_crossing_rate_std_dev': zero_crossing_rate_stats['std_dev']
    }

    return pd.DataFrame([features])

In [52]:
# Preprocessing input function
def preprocessing_input(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None)
        resampled_audio, new_sr = resampling(y, sr)
        denoised_audio = mmse_denoise(resampled_audio)
        normalized_audio = normalize_audio(denoised_audio)
        features = extract_audio_features(normalized_audio, new_sr)
        return features
    except Exception as e:
        print(f"Error in preprocessing {file_path}: {e}")
        return None


In [53]:
# Paths for data folders
Ravdess = "emotion_dataset/RAVDESS/"
Crema = "emotion_dataset/CREMA/"
Tess = "emotion_dataset/TESS/"
Savee = "emotion_dataset/SAVEE/"

data = []

# Process RAVDESS dataset
ravdess_directory_list = os.listdir(Ravdess)

for dir in ravdess_directory_list:
    actor = os.listdir(Ravdess + dir)
    for file in actor:
        part = file.split('.')[0].split('-')
        emotion = int(part[2])
        emotion_label = {
            1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad', 5: 'angry', 
            6: 'fear', 7: 'disgust', 8: 'surprise'
        }.get(emotion, 'Unknown')
        features = preprocessing_input(Ravdess + dir + '/' + file)
        if features is not None:
            features['Emotion'] = emotion_label
            data.append(features)  # Append the DataFrame
            print(f"Processed file {len(data)}: {file} (List size: {len(data)})")

# Process CREMA dataset
crema_directory_list = os.listdir(Crema)
for file in crema_directory_list:
    part = file.split('_')
    emotion = part[2]
    emotion_label = {
        'SAD': 'sad', 'ANG': 'angry', 'DIS': 'disgust', 
        'FEA': 'fear', 'HAP': 'happy', 'NEU': 'neutral'
    }.get(emotion, 'Unknown')
    features = preprocessing_input(Crema + file)
    if features is not None:
        features['Emotion'] = emotion_label
        data.append(features)
        print(f"Processed file {len(data)}: {file} (List size: {len(data)})")

# Process TESS dataset
tess_directory_list = os.listdir(Tess)
for dir in tess_directory_list:
    files = os.listdir(Tess + dir)
    for file in files:
        part = file.split('.')[0].split('_')[-1]
        emotion_label = 'surprise' if part == 'ps' else part
        features = preprocessing_input(Tess + dir + '/' + file)
        if features is not None:
            features['Emotion'] = emotion_label
            data.append(features)
            print(f"Processed file {len(data)}: {file} (List size: {len(data)})")

# Process SAVEE dataset
savee_directory_list = os.listdir(Savee)
for file in savee_directory_list:
    part = file.split('_')[1][:-6]
    emotion_label = {
        'a': 'angry', 'd': 'disgust', 'f': 'fear', 
        'h': 'happy', 'n': 'neutral', 'sa': 'sad'
    }.get(part, 'surprise')
    features = preprocessing_input(Savee + file)
    if features is not None:
        features['Emotion'] = emotion_label
        data.append(features)
        print(f"Processed file {len(data)}: {file} (List size: {len(data)})")

# Concatenate all feature DataFrames into one
emotion_training_data = pd.concat(data, ignore_index=True)

# Save to CSV
emotion_training_data.to_csv("emotion_training_data_whole_all2.csv", index=False)
print("Feature extraction and saving complete!")


Processed file 1: 03-01-01-01-01-01-01.wav (List size: 1)
Processed file 2: 03-01-01-01-01-02-01.wav (List size: 2)
Processed file 3: 03-01-01-01-02-01-01.wav (List size: 3)
Processed file 4: 03-01-01-01-02-02-01.wav (List size: 4)
Processed file 5: 03-01-02-01-01-01-01.wav (List size: 5)
Processed file 6: 03-01-02-01-01-02-01.wav (List size: 6)
Processed file 7: 03-01-02-01-02-01-01.wav (List size: 7)
Processed file 8: 03-01-02-01-02-02-01.wav (List size: 8)
Processed file 9: 03-01-02-02-01-01-01.wav (List size: 9)
Processed file 10: 03-01-02-02-01-02-01.wav (List size: 10)
Processed file 11: 03-01-02-02-02-01-01.wav (List size: 11)
Processed file 12: 03-01-02-02-02-02-01.wav (List size: 12)
Processed file 13: 03-01-03-01-01-01-01.wav (List size: 13)
Processed file 14: 03-01-03-01-01-02-01.wav (List size: 14)
Processed file 15: 03-01-03-01-02-01-01.wav (List size: 15)
Processed file 16: 03-01-03-01-02-02-01.wav (List size: 16)
Processed file 17: 03-01-03-02-01-01-01.wav (List size: 17

In [114]:
print(emotion_training_data)

       intensity_mean  intensity_min  intensity_max  intensity_std_dev  \
0           36.845562     -26.097175      86.350719          33.911723   
1           37.041570     -32.039217      85.532909          32.944943   
2           30.147223     -47.221381      84.551495          38.744592   
3           30.858774     -59.861087      83.862547          38.276569   
4           38.672611     -38.510437      86.974118          35.686907   
...               ...            ...            ...                ...   
12157       56.441557      12.544935      88.903158          23.198644   
12158       60.352141      16.750495      89.835305          18.353750   
12159       57.170718      16.740723      86.711545          20.853300   
12160       54.529843      12.267016      89.352187          22.371666   
12161       56.746448      10.641947      87.497163          20.514253   

       pitch_mean  pitch_min   pitch_max  pitch_std_dev       energy  \
0      111.659647  80.994733  157.75890